# Bangladesh National Election 2026 – Big Data Correlation & Predictive Analytics

## News Scrape Data Processing Pipeline

This section ingests all raw/standard/candidate files from `data_science/data/raw/news_scrape`, standardizes schema, cleans numeric/date fields, removes duplicates, and stores processed outputs in `data_science/data/processed`.

In [8]:
from pathlib import Path
import json
from datetime import datetime, UTC
import re

import numpy as np
import pandas as pd

PREFERRED_BASE_DIR = Path("/home/billy/X/Election-Dashboard")


def find_project_root() -> Path:
    expected_rel = Path("data_science/data/raw/news_scrape")

    if (PREFERRED_BASE_DIR / expected_rel).exists():
        return PREFERRED_BASE_DIR

    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / expected_rel).exists():
            return candidate

    return PREFERRED_BASE_DIR


BASE_DIR = find_project_root()
RAW_DIR = BASE_DIR / "data_science" / "data" / "raw" / "news_scrape"
PROCESSED_DIR = BASE_DIR / "data_science" / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, PROCESSED_DIR

(PosixPath('/home/billy/X/Election-Dashboard/data_science/data/raw/news_scrape'),
 PosixPath('/home/billy/X/Election-Dashboard/data_science/data/processed'))

In [9]:
STANDARD_COLUMNS = [
    "source", "source_file", "dataset_type", "url", "page_title", "scraped_at",
    "table_index", "row_index", "constituency", "candidates", "party",
    "votes", "turnout", "margin"
]


def detect_dataset_type(path: Path) -> str:
    name = path.stem.lower()
    for key in ["raw", "standard", "candidates"]:
        if f"_{key}_" in name or name.endswith(f"_{key}"):
            return key
    return "unknown"


def load_file(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() == ".json":
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        if isinstance(payload, list):
            if len(payload) == 0:
                return pd.DataFrame(columns=STANDARD_COLUMNS)
            df = pd.json_normalize(payload)
        elif isinstance(payload, dict):
            if "data" in payload and isinstance(payload["data"], list):
                df = pd.json_normalize(payload["data"])
            else:
                df = pd.json_normalize([payload])
        else:
            return pd.DataFrame(columns=STANDARD_COLUMNS)
    else:
        return pd.DataFrame(columns=STANDARD_COLUMNS)

    return df


def clean_numeric(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(r"[^\d\.-]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def standardize(df: pd.DataFrame, file_path: Path) -> pd.DataFrame:
    if df.empty:
        out = pd.DataFrame(columns=STANDARD_COLUMNS)
        return out

    rename_map = {
        "candidate": "candidates",
        "candidate_name": "candidates",
        "candidate_names": "candidates",
        "constituency_name": "constituency",
        "seat": "constituency",
    }
    df = df.rename(columns=rename_map)

    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    if df["source"].isna().all() or (df["source"].astype(str).str.strip() == "").all():
        df["source"] = file_path.stem.split("_")[0]

    df["source_file"] = file_path.name
    df["dataset_type"] = detect_dataset_type(file_path)

    text_cols = ["source", "source_file", "dataset_type", "url", "page_title", "constituency", "candidates", "party"]
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )

    for col in ["votes", "turnout", "margin", "row_index", "table_index"]:
        df[col] = clean_numeric(df[col])

    df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce", utc=True)

    out = df[STANDARD_COLUMNS].copy()
    out = out.drop_duplicates()
    return out

In [10]:
news_files = sorted([p for p in RAW_DIR.glob("*") if p.suffix.lower() in {".csv", ".json"}])

frames = []
file_level_stats = []

for path in news_files:
    try:
        raw_df = load_file(path)
        clean_df = standardize(raw_df, path)

        frames.append(clean_df)
        file_level_stats.append(
            {
                "source_file": path.name,
                "dataset_type": detect_dataset_type(path),
                "rows_in_raw": int(len(raw_df)),
                "rows_after_cleaning": int(len(clean_df)),
                "non_null_constituency": int(clean_df["constituency"].notna().sum()) if not clean_df.empty else 0,
                "non_null_candidates": int(clean_df["candidates"].notna().sum()) if not clean_df.empty else 0,
            }
        )
    except Exception as exc:
        file_level_stats.append(
            {
                "source_file": path.name,
                "dataset_type": detect_dataset_type(path),
                "rows_in_raw": 0,
                "rows_after_cleaning": 0,
                "non_null_constituency": 0,
                "non_null_candidates": 0,
                "error": str(exc),
            }
        )

processed_news = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=STANDARD_COLUMNS)

if len(processed_news):
    processed_news = processed_news.drop_duplicates(
        subset=["source_file", "row_index", "constituency", "candidates", "party", "votes", "margin"]
    )

# Add useful derived metric when both votes and turnout exist (safe zero-division handling)
processed_news["vote_share_pct"] = (
    processed_news["votes"] / processed_news["turnout"].replace(0, np.nan)
) * 100

run_stamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
news_out_csv = PROCESSED_DIR / "news_scrape_processed.csv"
news_out_json = PROCESSED_DIR / "news_scrape_processed.json"
news_out_csv_versioned = PROCESSED_DIR / f"news_scrape_processed_{run_stamp}.csv"
news_out_json_versioned = PROCESSED_DIR / f"news_scrape_processed_{run_stamp}.json"
stats_out_csv = PROCESSED_DIR / "news_scrape_file_stats.csv"

processed_news.to_csv(news_out_csv, index=False)
processed_news.to_csv(news_out_csv_versioned, index=False)
processed_news.to_json(news_out_json, orient="records", force_ascii=False, indent=2, date_format="iso")
processed_news.to_json(news_out_json_versioned, orient="records", force_ascii=False, indent=2, date_format="iso")

stats_df = pd.DataFrame(file_level_stats)
stats_df.to_csv(stats_out_csv, index=False)

print(f"Processed rows: {len(processed_news):,}")
print(f"Files scanned: {len(news_files)}")
print(f"Saved: {news_out_csv}")
print(f"Saved: {news_out_json}")
print(f"Saved: {stats_out_csv}")

if {"dataset_type", "source_file"}.issubset(stats_df.columns):
    stats_df.sort_values(["dataset_type", "source_file"]).head(20)
else:
    stats_df.head(20)

Processed rows: 4,926
Files scanned: 12
Saved: /home/billy/X/Election-Dashboard/data_science/data/processed/news_scrape_processed.csv
Saved: /home/billy/X/Election-Dashboard/data_science/data/processed/news_scrape_processed.json
Saved: /home/billy/X/Election-Dashboard/data_science/data/processed/news_scrape_file_stats.csv


In [11]:
processed_news.sample(min(10, len(processed_news)), random_state=42) if len(processed_news) else processed_news.head()

,source,source_file,dataset_type,url,page_title,scraped_at,table_index,row_index,constituency,candidates,party,votes,turnout,margin,vote_share_pct
1665,daily_star,daily_star_candidates_2026-02-20.csv,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Bogura 1,Kazi Rafiqul Islam,BNP,171440,NaN,NaN,NaN
3591,daily_star,daily_star_candidates_2026-02-20.json,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Natore 1,Md. Moazzem Hossain,Independent,0,NaN,NaN,NaN
2725,daily_star,daily_star_candidates_2026-02-20.json,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Lakshmipur 4,Md. Ashrafur Rahman Hafizullah,Bangladesh Jamaat-e-Islami,73756,NaN,NaN,NaN
2380,daily_star,daily_star_candidates_2026-02-20.json,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Jamalpur 3,Md. Mujibur Rahman Azadi,Bangladesh Jamaat-e-Islami,81430,NaN,NaN,NaN
676,daily_star,daily_star_candidates_2026-02-20.csv,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Cumilla 5,Tanzil Ahmed,Insaniat Biplob Bangladesh - Insaniat Biplob,0,NaN,NaN,NaN
290,daily_star,daily_star_candidates_2026-02-20.csv,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Bogura 2,Md. Rezaul Karim Talu,Independent,0,NaN,NaN,NaN
577,daily_star,daily_star_candidates_2026-02-20.csv,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Narayanganj 4,Anwar Hossain,Bangladesh Khilafat Majlis,0,NaN,NaN,NaN
2561,daily_star,daily_star_candidates_2026-02-20.json,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Narayanganj 3,Ariful Islam,Amar Bangladesh Party (AB Party),0,NaN,NaN,NaN
1519,daily_star,daily_star_candidates_2026-02-20.csv,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Bandarban 1,Saching Prue,BNP,141455,NaN,NaN,NaN
2194,daily_star,daily_star_candidates_2026-02-20.json,candidates,https://www.thedailystar.net/news/national-ele...,National Election 2026 | The Daily Star,2026-02-20 11:20:32.991818+00:00,NaN,NaN,Rangpur 6,Md. Saiful Islam,BNP,116919,NaN,NaN,NaN
